In [19]:
import pandas as pd
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

In [20]:
# 1. الاتصال بقاعدة البيانات
load_dotenv()
SERVER = os.getenv("DB_SERVER", "localhost")
DATABASE = os.getenv("DB_NAME", "HR_DW")
DRIVER = 'ODBC Driver 17 for SQL Server'
connection_string = f"mssql+pyodbc://@{SERVER}/{DATABASE}?driver={DRIVER}&trusted_connection=yes"
engine = create_engine(connection_string)

In [21]:
# 2. سحب البيانات (مع تطبيق فكرة استبعاد الإقالة والتقاعد)
query = """
SELECT 
    e.Age, e.Gender, e.MaritalStatus,
    f.TenureDays, f.PayZone, f.PerformanceScore, 
    f.EngagementScore, f.SatisfactionScore, f.WorkLifeBalanceScore, 
    f.TrainingCost, f.TrainingOutcome,
    f.AttritionFlag
FROM Fact_HR f
JOIN Dim_Employee e ON f.EmployeeKey = e.EmployeeKey
WHERE f.EmployeeKey != -1 

  AND (f.TerminationType IS NULL OR f.TerminationType NOT IN ('Involuntary', 'Retirement'))
"""
df = pd.read_sql(query, engine)

In [22]:
df.fillna({
    'Age': df['Age'].median(),
    'TenureDays': df['TenureDays'].median(),
    'TrainingCost': 0,
    'EngagementScore': 3,
    'SatisfactionScore': 3,
    'WorkLifeBalanceScore': 3,
    'PayZone': 'Unknown',
    'PerformanceScore': 'Not Evaluated', 
    'TrainingOutcome': 'None'
}, inplace=True)

,Age,Gender,MaritalStatus,TenureDays,PayZone,PerformanceScore,EngagementScore,SatisfactionScore,WorkLifeBalanceScore,TrainingCost,TrainingOutcome,AttritionFlag
0,69,Female,Married,2583,Zone A,Fully Meets,2,5,5,510.83,Failed,0
1,53,Female,Married,7,Zone A,Fully Meets,2,5,2,777.06,Incomplete,0
2,56,Male,Divorced,1469,Zone B,Fully Meets,2,4,5,145.99,Passed,0
3,67,Female,Widowed,97,Zone C,Fully Meets,5,2,1,838.07,Failed,0
4,45,Female,Married,250,Zone B,Fully Meets,5,2,2,758.18,Incomplete,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2230,39,Female,Single,1463,Zone A,Fully Meets,3,5,2,861.91,Failed,0
2231,80,Female,Widowed,687,Zone A,Exceeds,4,5,2,803.73,Incomplete,0
2232,27,Female,Widowed,1849,Zone C,Fully Meets,3,5,1,808.51,Failed,0
2233,80,Male,Single,2627,Zone A,Needs Improvement,2,4,1,629.16,Failed,0


In [23]:
# 4. تحويل النصوص لأرقام (One-Hot Encoding) وتقسيم الداتا
features = ['Age', 'Gender', 'MaritalStatus', 'TenureDays', 'PayZone', 
            'PerformanceScore', 'EngagementScore', 'SatisfactionScore', 
            'WorkLifeBalanceScore', 'TrainingCost', 'TrainingOutcome']
X = pd.get_dummies(df[features], drop_first=True)
y = df['AttritionFlag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [24]:
# 5. توحيد المقاييس (Scaling) - ضروري جداً لـ SVM و Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [25]:
# 6. إعداد الموديلات
stay_count = (y_train == 0).sum()
leave_count = (y_train == 1).sum()
tuned_weight = (stay_count / leave_count) * 0.6 

models = {
    "XGBoost (Production Model)": xgb.XGBClassifier(n_estimators=150, learning_rate=0.03, max_depth=4, subsample=0.8, scale_pos_weight=tuned_weight, random_state=42, eval_metric='logloss'),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=150, class_weight="balanced", random_state=42),
    "Support Vector Machine (SVM)": SVC(class_weight="balanced", random_state=42),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
}

In [26]:
# 7. تدريب الموديلات واستخراج النتائج
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results.append({
        "Algorithm": name,
        "Accuracy": f"{acc*100:.1f}%",
        "Recall (Flight Risk)": f"{rec*100:.1f}%",
        "F1-Score": f"{f1*100:.1f}%"
    })

In [27]:
# طباعة الجدول النهائي
results_df = pd.DataFrame(results)
print("\n🎯 Model Evaluation Results:")
print("="*60)
print(results_df.to_string(index=False))


🎯 Model Evaluation Results:
                   Algorithm Accuracy Recall (Flight Risk) F1-Score
  XGBoost (Production Model)    76.3%                86.8%    46.5%
    Random Forest Classifier    87.2%                 5.7%     9.5%
Support Vector Machine (SVM)    76.3%                84.9%    45.9%
         Logistic Regression    75.4%                88.7%    46.1%
